# PyHoloscope Tutorial 1: Inline Holography

In inline holography, light from a coherent light passes through a sample such as cells on a slide. A camera placed on the other side collects a diffraction pattern, showing interference between light scattered by the sample and light that passes straight through the slide. We can then use numerical methods to recover an image of the sample. This notebook demonstrates how to do this in Python using the PyHoloscope package. 


In [ ]:
from matplotlib import pyplot as plt
import sys; sys.path.append('..\src')   # Allows us to find PyHoloscope if not pip installed
import pyholoscope as pyh


An example hologram is saved as a tif file - this is what is captured by the camera. We can load this saved hologram using a convenience function in PyHoloscope, and then we display it:

In [ ]:
hologram = pyh.load_image(r"../test/integration_tests/test data/inline_example_holo.tif")

plt.figure(figsize = (3,3)); plt.imshow(hologram, cmap='gray')

We then create an instance of the Holo class, specifying that we are doing inline holography, as well as the wavelength of light and the camera pixel size. Finally, we also specify 'depth', the distance to numerically propagate light back, i.e. the distance from the sample to the camera.

In [ ]:
holo = pyh.Holo(
    mode=pyh.INLINE,    # For inline holography
    wavelength=630e-9,  # Light wavelength, m
    pixel_size=1e-6,    # Hologram physical pixel size, m
    depth=0.0130,       # Distance to refocus, m
)


Now we refocus the hologram.

In [ ]:
recon = holo.process(hologram)


This returns a complex array. We normally display the amplitude,

In [ ]:
plt.figure(figsize = (3,3)); plt.imshow(pyh.amplitude(recon), cmap='gray')

## Background Subtraction and Normalisation
An improved image is obtained if we first collect a background image, with nothing in between the light source and the camera. An example is provided as a second TIF file, which we load in:

In [ ]:
backFile = Path("../test/integration_tests/test data/inline_example_back.tif")
background = pyh.load_image(backFile)


We can use this in two ways. One is to subtract it from the hologram, creating what is known as a contrast hologram. The second is to divide the hologram by the background, correcting for variation in beam intensity across the image. Below we do both:

In [ ]:
holo = pyh.Holo(
    mode=pyh.INLINE,  # For inline holography
    wavelength=630e-9,  # Light wavelength, m
    pixel_size=1e-6,  # Hologram physical pixel size, m
    background=background,  # To subtract the background
    normalise=background,   # To normalise using background
    depth=0.0130,  # Distance to refocus, m
)

recon = holo.process(hologram)
plt.figure(figsize = (3,3)); plt.imshow(pyh.amplitude(recon), cmap='gray')


As a consequence of the background subtraction, the image now appears inverted in intensity, but we can invert again to obtain a brightfield style image:

In [ ]:
plt.figure(figsize = (3,3)); plt.imshow(pyh.invert(pyh.amplitude(recon)), cmap='gray')
